# Cargar el mostrador de Lente

Este notebook arma el catálogo ficticio de gafas y lo guarda en Qdrant.

Cada producto lleva precio, descripción y una foto gratuita de Unsplash. La foto y el texto se funden en **un** vector por proveedor: `google/gemini-embedding-2` (OpenRouter) y `embed-v4.0` (Cohere). Los dos viven en la misma colección, en vectores con nombre distinto, porque no comparten espacio.

Los documentos usan `search_document`. Las búsquedas del API usan `search_query`.

Corre las celdas desde `backend/` (o desde `backend/scripts/`). Hace falta Qdrant en `localhost:6333` y las claves en `backend/.env`. Volver a correrlo recrea la colección.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import httpx
from qdrant_client import AsyncQdrantClient

from app.config import Settings
from app.domains.catalog.domain.product import Product, StoredProduct, document_text
from app.domains.catalog.infrastructure.cohere_embedder import CohereEmbedder
from app.domains.catalog.infrastructure.gemini_embedder import GeminiEmbedder
from app.domains.catalog.infrastructure.qdrant_catalog import QdrantCatalog
from app.ssl import use_system_certificates

use_system_certificates()


def photo(photo_id: str) -> str:
    """Unsplash CDN url sized for the storefront and for embedding."""
    return f"https://images.unsplash.com/photo-{photo_id}?auto=format&fit=crop&w=1200&q=80"


## Catálogo

Dieciséis modelos en dos grupos: gafas de sol y armazones.


In [ ]:
PRODUCTS = [
    {
        "id": "sol-alba",
        "name": "Gafas De Sol Lente LN2104 Gris/Negro",
        "category": "gafas",
        "price": 89,
        "description": "Metal negro. Forma rectangular delgada. Unisex. Lunas grises, no polarizadas. Graduables.",
        "image_url": photo("1523884156331-22cc4f5df98d"),
    },
    {
        "id": "sol-bruma",
        "name": "Gafas De Sol Lente LN1840 Negro/Negro",
        "category": "gafas",
        "price": 118,
        "description": "Acetato negro. Forma agatada. Unisex. Lunas negras, no polarizadas.",
        "image_url": photo("1764740120242-77f9f5e3b63f"),
    },
    {
        "id": "sol-nube",
        "name": "Gafas De Sol Lente LN1622 Verde/Negro",
        "category": "gafas",
        "price": 96,
        "description": "Acetato negro. Unisex. Lunas verdes, no polarizadas. Protección UV.",
        "image_url": photo("1618677366787-9727aacca7ea"),
    },
    {
        "id": "sol-costa",
        "name": "Gafas De Sol Lente LN3310 Azul/Azul",
        "category": "gafas",
        "price": 132,
        "description": "Acetato azul. Unisex. Lunas azules, no polarizadas.",
        "image_url": photo("1701252374731-191f1b10a18b"),
    },
    {
        "id": "sol-aviador",
        "name": "Gafas De Sol Lente LN3548 Verde/Dorado",
        "category": "gafas",
        "price": 156,
        "description": "Metal dorado. Forma redonda. Unisex. Lunas verdes, no polarizadas. Graduables.",
        "image_url": photo("1511499767150-a48a237f0083"),
    },
    {
        "id": "sol-cobre",
        "name": "Gafas De Sol Lente LN3768 Café/Dorado",
        "category": "gafas",
        "price": 149,
        "description": "Metal dorado. Forma aviador. Unisex. Lunas café, no polarizadas.",
        "image_url": photo("1567473810954-507d59716c25"),
    },
    {
        "id": "sol-limon",
        "name": "Gafas De Sol Lente LN2198 Gris/Dorado",
        "category": "gafas",
        "price": 128,
        "description": "Metal dorado. Forma aviador. Unisex. Lunas grises, no polarizadas.",
        "image_url": photo("1759227922040-0ca4d3cbe42e"),
    },
    {
        "id": "dep-pista",
        "name": "Gafas De Sol Lente LN4187 Verde/Negro",
        "category": "gafas",
        "price": 138,
        "description": "Acetato negro. Unisex. Lunas verdes, no polarizadas. Protección UV.",
        "image_url": photo("1659506425360-f04a041e90df"),
    },
    {
        "id": "dep-viento",
        "name": "Gafas De Sol Lente LN3449 Café/Transparente",
        "category": "gafas",
        "price": 160,
        "description": "Acetato transparente. Unisex. Lunas café, no polarizadas.",
        "image_url": photo("1577803645773-f96470509666"),
    },
    {
        "id": "lec-tinta",
        "name": "Armazón Lente LN0842 Negro/Dorado",
        "category": "armazones",
        "price": 72,
        "description": "Frente negro, varillas doradas. Unisex. Armazón óptico con lunas transparentes graduables.",
        "image_url": photo("1602703522866-fb486308da5d"),
    },
    {
        "id": "lec-patio",
        "name": "Armazón Lente LN0715 Negro",
        "category": "armazones",
        "price": 68,
        "description": "Acetato negro. Unisex. Armazón óptico con lunas transparentes graduables.",
        "image_url": photo("1483412468200-72182dbbc544"),
    },
    {
        "id": "luz-pantalla",
        "name": "Armazón Lente LN5520 Negro",
        "category": "armazones",
        "price": 94,
        "description": "Metal negro. Unisex. Armazón óptico. Lunas transparentes graduables con filtro de luz azul.",
        "image_url": photo("1606357086272-eab87f3db598"),
    },
    {
        "id": "luz-teclado",
        "name": "Armazón Lente LN5488 Gris",
        "category": "armazones",
        "price": 88,
        "description": "Metal gris. Unisex. Armazón óptico. Lunas transparentes graduables con filtro de luz azul.",
        "image_url": photo("1614715838608-dd527c46231d"),
    },
    {
        "id": "luz-noche",
        "name": "Armazón Lente LN5614 Negro",
        "category": "armazones",
        "price": 102,
        "description": "Acetato negro. Unisex. Armazón óptico. Lunas transparentes graduables con filtro de luz azul.",
        "image_url": photo("1679237286210-e3ebb21b6f41"),
    },
    {
        "id": "mon-carey",
        "name": "Armazón Lente LN6401 Havana",
        "category": "armazones",
        "price": 180,
        "description": "Acetato havana. Forma clásica. Unisex. Armazón óptico con lunas transparentes graduables.",
        "image_url": photo("1603578119639-798b8413d8d7"),
    },
    {
        "id": "mon-oro",
        "name": "Armazón Lente LN7012 Dorado",
        "category": "armazones",
        "price": 210,
        "description": "Metal dorado. Unisex. Armazón óptico con lunas transparentes graduables.",
        "image_url": photo("1758552322632-ba288778c770"),
    },
]


## Embeddings y upsert

Descarga cada foto, pide el vector fusionado y los de texto e imagen, y reemplaza la colección `glasses`.


In [ ]:
async def seed() -> None:
    """Download each photo, embed it with both providers, and replace the collection."""
    settings = Settings()
    qdrant = AsyncQdrantClient(url=settings.qdrant_url)
    http = httpx.AsyncClient(timeout=60.0, follow_redirects=True)
    gemini = GeminiEmbedder(settings, http)
    cohere = CohereEmbedder(settings)
    repository = QdrantCatalog(qdrant, settings.collection_name, settings.embedding_dimension)
    stored: list[StoredProduct] = []
    try:
        for raw in PRODUCTS:
            product = Product(**raw)
            response = await http.get(product.image_url)
            response.raise_for_status()
            mime_type = response.headers.get("content-type", "image/jpeg").split(";")[0].strip()
            image = response.content
            text = document_text(product)
            gemini_vector = await gemini.embed_document(text, image, mime_type)
            gemini_text = await gemini.embed_document_text(text)
            gemini_image = await gemini.embed_document_image(image, mime_type)
            cohere_vector = await cohere.embed_document(text, image, mime_type)
            cohere_text = await cohere.embed_document_text(text)
            cohere_image = await cohere.embed_document_image(image, mime_type)
            stored.append(
                StoredProduct(
                    product=product,
                    gemini=gemini_vector,
                    cohere=cohere_vector,
                    gemini_text=gemini_text,
                    gemini_image=gemini_image,
                    cohere_text=cohere_text,
                    cohere_image=cohere_image,
                )
            )
            print(product.id, len(gemini_vector), len(cohere_vector))
        await repository.replace_all(stored)
        print(f"upserted {len(stored)} products")
    finally:
        await http.aclose()
        await qdrant.close()


await seed()
